# Word2Vec: From One-Hot Encoding to Custom Neural Network Embeddings
## Building Word2Vec from Scratch with PyTorch

This notebook demonstrates:
1. **One-Hot Encoding** - Traditional sparse representation
2. **Custom Neural Network** - Building Skip-gram model from scratch
3. **Training Loop** - Manual implementation of Word2Vec training
4. **Comparison** - One-hot vs. learned embeddings
5. **Visualization** - Understanding the learned representations

We'll use the classic sentence: **"The quick brown fox jumping over the lazy dog"**

In [ ]:
# Install required packages
!pip install -q torch matplotlib numpy pandas seaborn scikit-learn

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# PyTorch imports
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# Set random seed for reproducibility
np.random.seed(42)
torch.manual_seed(42)

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## Step 1: Prepare the Data

We'll start with our sample sentence and create a small corpus for training.

In [ ]:
# Sample sentence
sample_sentence = "The quick brown fox jumping over the lazy dog"

print("Original Sentence:")
print(f"'{sample_sentence}'")
print(f"\nLength: {len(sample_sentence)} characters")

In [ ]:
# Tokenize the sentence
tokens = sample_sentence.lower().split()

print("Tokenized:")
print(tokens)
print(f"\nNumber of tokens: {len(tokens)}")
print(f"Unique words: {len(set(tokens))}")

In [ ]:
# Create vocabulary
vocabulary = sorted(set(tokens))
vocab_size = len(vocabulary)

# Create word to index and index to word mappings
word_to_idx = {word: idx for idx, word in enumerate(vocabulary)}
idx_to_word = {idx: word for word, idx in word_to_idx.items()}

print("Vocabulary (alphabetically sorted):")
for idx, word in enumerate(vocabulary):
    print(f"{idx}: {word}")

print(f"\nVocabulary size: {vocab_size}")

## Step 2: One-Hot Encoding

One-hot encoding represents each word as a vector of length V (vocabulary size), where:
- All elements are 0, except
- One element is 1 at the position corresponding to the word

**Example:** If vocabulary = [brown, dog, fox, ...], then:
- 'brown' = [1, 0, 0, 0, 0, 0, 0, 0]
- 'dog' = [0, 1, 0, 0, 0, 0, 0, 0]
- 'fox' = [0, 0, 1, 0, 0, 0, 0, 0]

In [ ]:
def create_one_hot(word, word_to_idx, vocab_size):
    """
    Create one-hot encoding for a word
    
    Args:
        word: string, the word to encode
        word_to_idx: dict, mapping from word to index
        vocab_size: int, size of vocabulary
    
    Returns:
        numpy array of shape (vocab_size,) with one-hot encoding
    """
    one_hot = np.zeros(vocab_size)
    idx = word_to_idx[word]
    one_hot[idx] = 1
    return one_hot

In [ ]:
# Create one-hot encodings for all words in our sentence
one_hot_encodings = {}

for word in vocabulary:
    one_hot_encodings[word] = create_one_hot(word, word_to_idx, vocab_size)

print("One-Hot Encodings:")
print("="*50)
for word in vocabulary[:3]:  # Show first 3 words
    encoding = one_hot_encodings[word]
    print(f"\n'{word}':")
    print(f"Vector: {encoding}")
    print(f"Shape: {encoding.shape}")
    print(f"Index of '1': {np.argmax(encoding)}")

In [ ]:
# Visualize one-hot encodings as a matrix
one_hot_matrix = np.array([one_hot_encodings[word] for word in vocabulary])

plt.figure(figsize=(10, 8))
plt.imshow(one_hot_matrix, cmap='YlOrRd', aspect='auto')
plt.colorbar(label='Value')
plt.xlabel('Dimension', fontsize=12)
plt.ylabel('Word', fontsize=12)
plt.title('One-Hot Encoding Matrix\n(Each row is a word, each column is a dimension)', 
          fontsize=14, fontweight='bold')
plt.yticks(range(vocab_size), vocabulary)
plt.xticks(range(vocab_size), range(vocab_size))
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"One-hot matrix shape: {one_hot_matrix.shape}")
print(f"Total elements: {one_hot_matrix.size}")
print(f"Non-zero elements: {np.count_nonzero(one_hot_matrix)}")
print(f"Sparsity: {(1 - np.count_nonzero(one_hot_matrix) / one_hot_matrix.size) * 100:.1f}%")

### Problems with One-Hot Encoding

Let's demonstrate the key limitations:

In [ ]:
# Problem 1: No semantic similarity
def cosine_similarity(v1, v2):
    """Calculate cosine similarity between two vectors"""
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

# Compare semantic pairs
word_pairs = [
    ('fox', 'dog'),      # Both animals
    ('quick', 'lazy'),   # Both adjectives (opposites)
    ('brown', 'fox'),    # Unrelated
    ('the', 'over')      # Both function words
]

print("Cosine Similarity using One-Hot Encoding:")
print("="*50)
for word1, word2 in word_pairs:
    vec1 = one_hot_encodings[word1]
    vec2 = one_hot_encodings[word2]
    similarity = cosine_similarity(vec1, vec2)
    print(f"similarity('{word1}', '{word2}') = {similarity:.4f}")

print("\n💡 Problem: All word pairs have similarity = 0.0")
print("   One-hot encoding treats all words as equally different!")

## Step 3: Create Training Data for Skip-gram

Skip-gram model learns to predict context words given a target word.
For each word, we create training pairs (target_word, context_word).

In [ ]:
def create_skipgram_dataset(tokens, word_to_idx, window_size=2):
    """
    Create skip-gram training pairs
    
    Args:
        tokens: list of words
        word_to_idx: dict mapping words to indices
        window_size: context window size
    
    Returns:
        target_indices: list of target word indices
        context_indices: list of context word indices
    """
    target_indices = []
    context_indices = []
    
    for i, target_word in enumerate(tokens):
        target_idx = word_to_idx[target_word]
        
        # Get context words within window
        start = max(0, i - window_size)
        end = min(len(tokens), i + window_size + 1)
        
        for j in range(start, end):
            if j != i:  # Skip the target word itself
                context_word = tokens[j]
                context_idx = word_to_idx[context_word]
                target_indices.append(target_idx)
                context_indices.append(context_idx)
    
    return target_indices, context_indices

# Create expanded corpus for better training
corpus = [
    "the quick brown fox jumping over the lazy dog",
    "the quick brown fox",
    "the lazy dog",
    "quick brown fox jumping",
    "jumping over the dog",
    "brown fox and lazy dog",
    "the fox over the dog",
    "quick fox jumping over"
]

# Tokenize entire corpus
all_tokens = []
for sentence in corpus:
    all_tokens.extend(sentence.lower().split())

print(f"Total tokens in corpus: {len(all_tokens)}")
print(f"Unique words: {len(set(all_tokens))}")

# Create training data
window_size = 2
target_indices, context_indices = create_skipgram_dataset(all_tokens, word_to_idx, window_size)

print(f"\nTraining pairs created: {len(target_indices)}")
print(f"Window size: {window_size}")

# Show some examples
print("\nExample training pairs:")
print("="*50)
for i in range(min(10, len(target_indices))):
    target = idx_to_word[target_indices[i]]
    context = idx_to_word[context_indices[i]]
    print(f"Target: '{target}' → Context: '{context}'")

## Step 4: Build Custom Skip-gram Neural Network

We'll build a simple neural network with:
- **Input Layer**: One-hot encoded target word (vocab_size)
- **Hidden Layer**: Embedding layer (vocab_size × embedding_dim)
- **Output Layer**: Predicts context word (embedding_dim × vocab_size)

In [ ]:
class SkipGramModel(nn.Module):
    """
    Custom Skip-gram Neural Network
    
    Architecture:
    Input (vocab_size) → Embedding (embedding_dim) → Output (vocab_size)
    """
    def __init__(self, vocab_size, embedding_dim):
        super(SkipGramModel, self).__init__()
        self.vocab_size = vocab_size
        self.embedding_dim = embedding_dim
        
        # Embedding layer (this is our word vectors)
        self.embeddings = nn.Embedding(vocab_size, embedding_dim)
        
        # Output layer
        self.linear = nn.Linear(embedding_dim, vocab_size)
        
        # Initialize weights
        self.embeddings.weight.data.uniform_(-0.5 / embedding_dim, 0.5 / embedding_dim)
        self.linear.weight.data.uniform_(-0.5 / vocab_size, 0.5 / vocab_size)
    
    def forward(self, target_indices):
        """
        Forward pass
        
        Args:
            target_indices: tensor of target word indices
        
        Returns:
            log probabilities for context words
        """
        # Get embeddings for target words
        embeds = self.embeddings(target_indices)  # (batch_size, embedding_dim)
        
        # Project to vocabulary space
        output = self.linear(embeds)  # (batch_size, vocab_size)
        
        # Apply log softmax for numerical stability
        log_probs = torch.log_softmax(output, dim=1)
        
        return log_probs
    
    def get_embedding(self, word_idx):
        """
        Get embedding vector for a word
        
        Args:
            word_idx: index of the word
        
        Returns:
            embedding vector
        """
        return self.embeddings.weight[word_idx].detach().cpu().numpy()

# Model hyperparameters
embedding_dim = 50
learning_rate = 0.01
epochs = 1000

# Initialize model
model = SkipGramModel(vocab_size, embedding_dim).to(device)

print("Custom Skip-gram Model Architecture:")
print("="*50)
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters())}")
print(f"Embedding parameters: {vocab_size * embedding_dim}")
print(f"Output layer parameters: {embedding_dim * vocab_size + vocab_size}")

## Step 5: Create Dataset and DataLoader

In [ ]:
class SkipGramDataset(Dataset):
    """
    PyTorch Dataset for Skip-gram training
    """
    def __init__(self, target_indices, context_indices):
        self.target_indices = torch.LongTensor(target_indices)
        self.context_indices = torch.LongTensor(context_indices)
    
    def __len__(self):
        return len(self.target_indices)
    
    def __getitem__(self, idx):
        return self.target_indices[idx], self.context_indices[idx]

# Create dataset and dataloader
dataset = SkipGramDataset(target_indices, context_indices)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

print(f"Dataset size: {len(dataset)}")
print(f"Number of batches: {len(dataloader)}")
print(f"Batch size: 32")

## Step 6: Train the Custom Neural Network

We'll train using Negative Log-Likelihood Loss (equivalent to cross-entropy).

In [ ]:
# Loss function and optimizer
criterion = nn.NLLLoss()  # Negative Log-Likelihood Loss
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# Training loop
print("Training Custom Skip-gram Neural Network...")
print("="*70)

loss_history = []

for epoch in range(epochs):
    total_loss = 0
    
    for batch_targets, batch_contexts in dataloader:
        # Move to device
        batch_targets = batch_targets.to(device)
        batch_contexts = batch_contexts.to(device)
        
        # Zero gradients
        optimizer.zero_grad()
        
        # Forward pass
        log_probs = model(batch_targets)
        
        # Calculate loss
        loss = criterion(log_probs, batch_contexts)
        
        # Backward pass
        loss.backward()
        
        # Update weights
        optimizer.step()
        
        total_loss += loss.item()
    
    avg_loss = total_loss / len(dataloader)
    loss_history.append(avg_loss)
    
    # Print progress
    if (epoch + 1) % 100 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}")

print("\n✓ Training completed!")

In [ ]:
# Plot training loss
plt.figure(figsize=(10, 5))
plt.plot(loss_history, linewidth=2)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.title('Training Loss over Time', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Initial loss: {loss_history[0]:.4f}")
print(f"Final loss: {loss_history[-1]:.4f}")
print(f"Loss reduction: {(1 - loss_history[-1]/loss_history[0]) * 100:.1f}%")

## Step 7: Extract Learned Embeddings

The embeddings are now stored in the model's embedding layer.

In [ ]:
# Extract embeddings for all words
model.eval()  # Set to evaluation mode

word_embeddings = {}
for word in vocabulary:
    idx = word_to_idx[word]
    embedding = model.get_embedding(idx)
    word_embeddings[word] = embedding

print("Learned Word Embeddings:")
print("="*50)
for word in vocabulary[:3]:  # Show first 3 words
    embedding = word_embeddings[word]
    print(f"\n'{word}':")
    print(f"Vector (first 10 dims): {embedding[:10]}")
    print(f"Shape: {embedding.shape}")
    print(f"L2 norm: {np.linalg.norm(embedding):.4f}")

## Step 8: Compare Word Similarities

Now let's see if our custom model learned semantic relationships!

In [ ]:
# Compare semantic pairs using learned embeddings
print("Cosine Similarity using Custom Word2Vec Embeddings:")
print("="*50)
for word1, word2 in word_pairs:
    vec1 = word_embeddings[word1]
    vec2 = word_embeddings[word2]
    similarity = cosine_similarity(vec1, vec2)
    print(f"similarity('{word1}', '{word2}') = {similarity:.4f}")

print("\n💡 Improvement: Non-zero similarities!")
print("   The model learned semantic relationships from context.")

In [ ]:
# Find most similar words for each word
def find_most_similar(target_word, word_embeddings, word_to_idx, top_k=3):
    """
    Find most similar words to target word
    """
    target_vec = word_embeddings[target_word]
    
    similarities = {}
    for word, vec in word_embeddings.items():
        if word != target_word:
            sim = cosine_similarity(target_vec, vec)
            similarities[word] = sim
    
    # Sort by similarity
    sorted_words = sorted(similarities.items(), key=lambda x: x[1], reverse=True)
    return sorted_words[:top_k]

print("Most Similar Words (Top 3):")
print("="*50)
for word in ['fox', 'quick', 'the']:
    similar_words = find_most_similar(word, word_embeddings, word_to_idx, top_k=3)
    print(f"\n'{word}' is most similar to:")
    for sim_word, score in similar_words:
        print(f"  {sim_word}: {score:.4f}")

## Step 9: Visualize Embeddings

In [ ]:
# Create embedding matrix for visualization
embedding_matrix = np.array([word_embeddings[word] for word in vocabulary])

# Reduce to 2D using PCA
pca = PCA(n_components=2)
embeddings_2d = pca.fit_transform(embedding_matrix)

# Plot
plt.figure(figsize=(12, 8))
plt.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], s=200, alpha=0.6, c='steelblue')

# Add labels
for i, word in enumerate(vocabulary):
    plt.annotate(word, 
                xy=(embeddings_2d[i, 0], embeddings_2d[i, 1]),
                xytext=(5, 5),
                textcoords='offset points',
                fontsize=11,
                fontweight='bold')

plt.xlabel('First Principal Component', fontsize=12)
plt.ylabel('Second Principal Component', fontsize=12)
plt.title('Custom Word2Vec Embeddings Visualization (PCA)', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Variance explained by PC1: {pca.explained_variance_ratio_[0]:.1%}")
print(f"Variance explained by PC2: {pca.explained_variance_ratio_[1]:.1%}")
print(f"Total variance explained: {sum(pca.explained_variance_ratio_):.1%}")

## Step 10: Compare One-Hot vs Custom Embeddings

In [ ]:
# Encode sentence using both methods
sentence_onehot = np.mean([one_hot_encodings[word] 
                           for word in tokens], axis=0)

sentence_custom = np.mean([word_embeddings[word] 
                           for word in tokens], axis=0)

print("Sentence Encoding Comparison:")
print("="*70)
print("\nOne-Hot Encoding (averaged):")
print(f"  Dimension: {sentence_onehot.shape[0]}")
print(f"  Non-zero elements: {np.count_nonzero(sentence_onehot)}")
print(f"  Sparsity: {(1 - np.count_nonzero(sentence_onehot) / sentence_onehot.size) * 100:.1f}%")

print("\nCustom Word2Vec Encoding (averaged):")
print(f"  Dimension: {sentence_custom.shape[0]}")
print(f"  Non-zero elements: {np.count_nonzero(sentence_custom)}")
print(f"  Sparsity: {(1 - np.count_nonzero(sentence_custom) / sentence_custom.size) * 100:.1f}%")

In [ ]:
# Comparison table
comparison_data = {
    'Method': ['One-Hot', 'Custom Neural Network'],
    'Dimensions': [sentence_onehot.shape[0], sentence_custom.shape[0]],
    'Sparsity': [
        f"{(1 - np.count_nonzero(sentence_onehot) / sentence_onehot.size) * 100:.1f}%",
        f"{(1 - np.count_nonzero(sentence_custom) / sentence_custom.size) * 100:.1f}%"
    ],
    'Captures Semantics': ['No', 'Yes'],
    'Training Required': ['No', 'Yes (1000 epochs)']
}

df_comparison = pd.DataFrame(comparison_data)
print("\nComparison Summary:")
print("="*70)
print(df_comparison.to_string(index=False))

print("\n💡 Key Takeaway:")
print(f"   Custom model uses {sentence_custom.shape[0]} dimensions vs {sentence_onehot.shape[0]} for one-hot")
print(f"   That's a {sentence_onehot.shape[0] / sentence_custom.shape[0]:.1f}x reduction!")
print(f"   Plus, it captures semantic meaning through neural network training!")

## Step 11: Understanding the Neural Network Architecture

Let's visualize how the network works.

In [ ]:
print("Custom Skip-gram Neural Network Architecture")
print("="*70)
print("\nLayer Structure:")
print("  1. Input Layer: One-hot encoded word (vocab_size = 9)")
print("  2. Embedding Layer: Dense representation (embedding_dim = 50)")
print("     - This is our WORD VECTOR layer")
print("     - Shape: (9, 50)")
print("     - Each row is a word's embedding")
print("  3. Output Layer: Predict context word (vocab_size = 9)")
print("     - Shape: (50, 9)")
print("     - Uses softmax to get probabilities")

print("\nTraining Process:")
print("  1. Input: target word 'fox' (one-hot: [0,0,1,0,0,0,0,0,0])")
print("  2. Embedding: Look up 'fox' vector from embedding matrix")
print("  3. Output: Predict context words ('brown', 'jumping', etc.)")
print("  4. Loss: Compare prediction with actual context word")
print("  5. Backprop: Update embeddings to minimize loss")

print("\nWhat Makes It 'Learn'?")
print("  - Words appearing in similar contexts get similar embeddings")
print("  - 'fox' and 'dog' both appear near 'the', 'brown', 'lazy'")
print("  - So their embeddings become similar!")

## Step 12: Summary

In [ ]:
print("\n" + "="*70)
print("SUMMARY: One-Hot vs Custom Neural Network Embeddings")
print("="*70)

summary_data = {
    'Aspect': [
        'Method',
        'Dimensionality',
        'Density',
        'Semantic Meaning',
        'Memory Efficiency',
        'Training Required',
        'Customizable',
        'Interpretability'
    ],
    'One-Hot Encoding': [
        'Simple lookup',
        f'{vocab_size} (= vocab size)',
        f'{(1 - np.count_nonzero(one_hot_matrix) / one_hot_matrix.size) * 100:.0f}% sparse',
        'No',
        'Poor - mostly zeros',
        'No',
        'No',
        'High - clear mapping'
    ],
    'Custom Neural Network': [
        'Learned embeddings',
        '50 (user-defined)',
        '~0% sparse (dense)',
        'Yes - from context',
        'Excellent - compact',
        'Yes - backpropagation',
        'Yes - full control',
        'Medium - learned features'
    ]
}

df_summary = pd.DataFrame(summary_data)
print("\n" + df_summary.to_string(index=False))

print("\n" + "="*70)
print("KEY INSIGHTS FROM CUSTOM IMPLEMENTATION")
print("="*70)
print("\n1. NEURAL NETWORK ARCHITECTURE:")
print("   - Embedding layer stores word vectors")
print("   - Output layer predicts context")
print("   - Skip-gram: target → context")

print("\n2. TRAINING PROCESS:")
print("   - Forward pass: compute predictions")
print("   - Loss: NLL loss (cross-entropy)")
print("   - Backward pass: update embeddings via gradient descent")

print("\n3. BENEFITS OF CUSTOM IMPLEMENTATION:")
print("   - Full control over architecture")
print("   - Can modify loss function")
print("   - Can add regularization, dropout, etc.")
print("   - Better understanding of internals")

print("\n4. COMPARISON WITH GENSIM:")
print("   - Gensim: Optimized C implementation, faster")
print("   - Custom: More flexible, educational")
print("   - Both learn same skip-gram objective")

print("\n" + "="*70)

## Exercises

Try these to deepen your understanding:

1. **Modify Architecture**:
   - Add a second hidden layer
   - Try different activation functions
   - Add dropout for regularization

2. **Experiment with Hyperparameters**:
   - Different embedding dimensions (10, 100, 200)
   - Different learning rates (0.001, 0.01, 0.1)
   - Different window sizes (1, 3, 5)

3. **Try CBOW Instead of Skip-gram**:
   - Reverse the prediction: context → target
   - Compare results with Skip-gram

4. **Add Negative Sampling**:
   - Sample random "negative" words
   - Train to distinguish real vs fake context
   - This is how real Word2Vec works!

5. **Visualize Training Dynamics**:
   - Plot embeddings at different epochs
   - Watch how words cluster over time